# Day 26: Build a "Hierarchical Chunking" strategy

## Core Theory (Just-in-Time)

When building Retrieval-Augmented Generation (RAG) systems, you often face a fundamental trade-off when deciding on chunk size:
* **Small Chunks (e.g., 200 tokens):** Yield highly specific and accurate vector similarity matches, but they lack the surrounding context. When passed to the LLM, the LLM might struggle to synthesize a comprehensive answer.
* **Large Chunks (e.g., 2000 tokens):** Provide excellent context for the LLM, but they contain too much noise. The vector embeddings become diluted, leading to less precise retrieval.

**Hierarchical Chunking (also known as Parent-Child Chunking)** resolves this tension by giving you the best of both worlds:
1. **Parent Chunks:** Split documents into large "parent" chunks.
2. **Child Chunks:** Split those parent chunks into smaller "child" chunks.
3. **Storage:** Embed and store the *child* chunks in the vector database. Crucially, each child chunk maintains a reference (ID) to its parent chunk. The parent chunks are stored in a standard key-value document store.
4. **Retrieval:** During a query, you search against the *child* chunks for precise matching. Once the most relevant child chunks are identified, the system looks up their corresponding *parent* chunks and passes those larger context blocks to the LLM.

**AI Security Implications:**
* **PII & Data Leakage:** When using hierarchical chunking, a query might match a benign child chunk, but the returned parent chunk could contain sensitive PII. Ensure redaction is performed on the parent chunks *before* they are returned to the LLM.
* **Prompt Injection in Context:** Larger context windows mean more surface area for adversarial text hidden in documents to act as prompt injections. Ensure proper fallbacks and system prompt boundaries are in place.
* **Resource Exhaustion:** Overly large parent chunks can cause context window overflows, leading to increased API costs or denial-of-service (DoS) if the model fails. Always bound parent chunk sizes.


## Code Implementation


### Basic Implementation
Isolates the core concept of setting up a `ParentDocumentRetriever` with minimal boilerplate.

In [1]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_core.stores import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_openai import OpenAIEmbeddings

if os.environ.get("OPENAI_API_KEY") and os.environ.get("OPENAI_API_KEY") != "sk-dummy":
    # Minimal setup
    docs = [Document(page_content="AI Engineering is hard but rewarding. " * 50)]

    client = QdrantClient(location=":memory:")
    client.create_collection(
        collection_name="basic_collection",
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
    )
    vectorstore = QdrantVectorStore(
        client=client, collection_name="basic_collection", embedding=OpenAIEmbeddings()
    )
    store = InMemoryStore()

    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=500)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=100)

    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=store,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )

    retriever.add_documents(docs)
    res = retriever.invoke("AI Engineering")
    print(f"Retrieved chunks: {len(res)}, Length of first: {len(res[0].page_content)}")
else:
    print("OPENAI_API_KEY not set or is dummy. Skipping execution.")


OPENAI_API_KEY not set or is dummy. Skipping execution.


### Medium Implementation
Emphasizes clean OOP, state management, and clear interactions between components.

In [2]:
import os
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_core.stores import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_openai import OpenAIEmbeddings

class HierarchicalRetrieverManager:
    def __init__(self, collection_name: str = "medium_collection"):
        self.collection_name = collection_name
        self._init_components()
        
    def _init_components(self):
        # Qdrant Client
        self.client = QdrantClient(location=":memory:")
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
        )
        # Vectorstore
        self.vectorstore = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=OpenAIEmbeddings()
        )
        # Docstore
        self.docstore = InMemoryStore()
        # Retriever
        self.retriever = ParentDocumentRetriever(
            vectorstore=self.vectorstore,
            docstore=self.docstore,
            child_splitter=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20),
            parent_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0),
        )
        
    def index_documents(self, documents: List[Document]):
        self.retriever.add_documents(documents)
        
    def query(self, query: str) -> List[Document]:
        return self.retriever.invoke(query)

if __name__ == "__main__":
    if os.environ.get("OPENAI_API_KEY") and os.environ.get("OPENAI_API_KEY") != "sk-dummy":
        manager = HierarchicalRetrieverManager()
        docs = [Document(page_content="Clean OOP makes AI engineering much more maintainable. " * 30)]
        manager.index_documents(docs)
        results = manager.query("OOP")
        print(f"Medium: Retrieved {len(results)} parent documents.")
    else:
        print("OPENAI_API_KEY not set or is dummy. Skipping execution.")


OPENAI_API_KEY not set or is dummy. Skipping execution.


### Advanced Implementation
Production-grade implementation with strict type hinting, docstrings, error handling, PII redaction (AI Security), and exact import syntax.

In [3]:
import os
import re
import logging
from typing import List, Optional
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_core.stores import InMemoryStore, BaseStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_openai import OpenAIEmbeddings

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecureHierarchicalRetriever:
    """
    A production-grade Hierarchical Retriever that includes basic PII redaction 
    and robust error handling mechanisms.
    """
    def __init__(
        self, 
        collection_name: str = "advanced_collection",
        embeddings: Optional[Embeddings] = None
    ) -> None:
        self.collection_name = collection_name
        
        if embeddings is None:
            if not os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY") == "sk-dummy":
                raise ValueError("Valid OPENAI_API_KEY not found in environment, cannot initialize embeddings.")
            self.embeddings = OpenAIEmbeddings()
        else:
            self.embeddings = embeddings
            
        self.client: QdrantClient = self._initialize_qdrant()
        self.vectorstore: QdrantVectorStore = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=self.embeddings
        )
        self.docstore: BaseStore = InMemoryStore()
        
        self.retriever: ParentDocumentRetriever = ParentDocumentRetriever(
            vectorstore=self.vectorstore,
            docstore=self.docstore,
            child_splitter=RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20),
            parent_splitter=RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=0),
        )
        
    def _initialize_qdrant(self) -> QdrantClient:
        """Initializes the Qdrant client and creates the collection."""
        try:
            client = QdrantClient(location=":memory:")
            client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
            )
            return client
        except Exception as e:
            logger.error(f"Failed to initialize Qdrant: {e}")
            raise RuntimeError(f"Database initialization failed: {e}")
            
    def _redact_pii(self, text: str) -> str:
        """
        A basic AI Security measure to redact sensitive data (e.g., Email addresses) 
        before returning the parent chunks.
        """
        email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
        return re.sub(email_pattern, '[REDACTED_EMAIL]', text)

    def index(self, documents: List[Document]) -> None:
        """Indexes the documents into the hierarchical structure."""
        try:
            logger.info(f"Indexing {len(documents)} documents.")
            self.retriever.add_documents(documents)
            logger.info("Indexing complete.")
        except Exception as e:
            logger.error(f"Error during document indexing: {e}")
            raise
            
    def search(self, query: str) -> List[Document]:
        """
        Queries the retriever and applies PII redaction as a security fallback.
        """
        try:
            logger.info(f"Querying for: {query}")
            results = self.retriever.invoke(query)
            
            # Apply AI Security fallback: Redact PII from the large context blocks
            sanitized_results = []
            for doc in results:
                sanitized_content = self._redact_pii(doc.page_content)
                sanitized_results.append(Document(page_content=sanitized_content, metadata=doc.metadata))
                
            return sanitized_results
        except Exception as e:
            logger.error(f"Error during search: {e}")
            return []

if __name__ == "__main__":
    if os.environ.get("OPENAI_API_KEY") and os.environ.get("OPENAI_API_KEY") != "sk-dummy":
        secure_retriever = SecureHierarchicalRetriever()
        docs = [
            Document(page_content="Here is some context. Contact me at john.doe@secret-domain.com for more information. " * 15)
        ]
        secure_retriever.index(docs)
        
        results = secure_retriever.search("Contact information")
        if results:
            print(f"Advanced Result Preview: {results[0].page_content[:150]}...")
    else:
        print("OPENAI_API_KEY not set or is dummy. Skipping advanced execution.")


OPENAI_API_KEY not set or is dummy. Skipping advanced execution.


## Practical Lab / Homework

**Your Task:**
1. Read a real document (like a small PDF or text file) using a LangChain document loader.
2. Implement the `ParentDocumentRetriever` using Qdrant as the vector store and `InMemoryStore` as the docstore.
3. Query the retriever with a specific question and verify that the returned chunk is the *parent* chunk (e.g., check the string length or print the output).
4. **AI Security Challenge:** Implement a fallback mechanism if the vector store fails to return results, or add a redaction filter for PII before passing the parent chunk back.
5. **Bonus:** Try modifying the `parent_splitter` to return full documents instead of chunks (by omitting the `parent_splitter` argument or setting its chunk size to be very large), and observe how the results change.

**Video Walkthrough:**
Please record a brief (2-3 minute) async video walkthrough (e.g., Loom) explaining your design decisions, specifically focusing on how you managed the relationship between parent and child chunks, and how you addressed the AI Security challenge.


In [4]:
# LAB TASK SOLUTION
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from langchain_core.stores import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os

api_key_set = os.environ.get("OPENAI_API_KEY") and os.environ.get("OPENAI_API_KEY") != "sk-dummy"

if api_key_set:
    # 1. Create a document directly for the lab task
    lab_docs = [
        Document(
            page_content="""
            Chapter 1: The Beginning. The quick brown fox jumps over the lazy dog.
            This is a very important sentence about the fox.
            
            Chapter 2: The Middle. The dog was not amused. It barked loudly.
            
            Chapter 3: The End. They all went to sleep.
            """
        )
    ]
    
    # 2. Setup the retriever using Qdrant and InMemoryStore
    lab_client = QdrantClient(location=":memory:")
    lab_client.create_collection(
        collection_name="lab_collection",
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
    )
    
    lab_vectorstore = QdrantVectorStore(
        client=lab_client,
        collection_name="lab_collection",
        embedding=OpenAIEmbeddings()
    )
    
    lab_store = InMemoryStore()
    
    lab_parent_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)
    lab_child_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
    
    lab_retriever = ParentDocumentRetriever(
        vectorstore=lab_vectorstore,
        docstore=lab_store,
        child_splitter=lab_child_splitter,
        parent_splitter=lab_parent_splitter,
    )
    
    lab_retriever.add_documents(lab_docs)
    
    # 3. Query the retriever
    query = "What did the dog do?"
    results = lab_retriever.invoke(query)
    
    # 4. Print results to verify you got the parent context
    print("LAB RESULTS:")
    for idx, res in enumerate(results):
        print(f"Result {idx + 1} (Length {len(res.page_content)}):\n{res.page_content.strip()}\n")
else:
    print("LAB RESULTS: Cannot execute query without valid OPENAI_API_KEY")


LAB RESULTS: Cannot execute query without valid OPENAI_API_KEY


## Common Pitfalls

1. **Storage De-synchronization:** When deleting documents, you must remember to delete *both* the parent document from the DocStore and the associated child embeddings from the VectorStore. If they get out of sync, you may retrieve child IDs that don't map to any parent.
2. **Overlap Tuning:** While child chunks should have overlap to maintain context across chunk boundaries, parent chunks often need less or no overlap, as they are already large. Tuning these overlap parameters incorrectly can lead to duplicating large amounts of text in your DocStore.
3. **Retrieval Latency:** Hierarchical chunking requires a two-step retrieval process (Vector search -> DocStore lookup). If your DocStore is slow (e.g., a poorly indexed relational database or high-latency network store), this will significantly slow down your RAG pipeline. Redis or fast key-value stores are recommended for production DocStores.
4. **Memory Exhaustion:** Using `InMemoryStore` is great for local testing and notebooks, but it is not ephemeral between restarts. Always use a persistent store (like RedisStore or a persistent database) in production to avoid losing your parent mappings.



## Reference Links
* [LangChain: ParentDocumentRetriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)
* [Qdrant: Hierarchical Navigable Small World (HNSW)](https://qdrant.tech/articles/hnsw/)
* [OWASP Top 10 for LLM Applications (For PII/Data Leakage)](https://owasp.org/www-project-top-10-for-large-language-model-applications/)
